In [ ]:
# =============================================================================
# Word2Vec 词嵌入训练 - 数据预处理
# =============================================================================
# Word2Vec是Google提出的词嵌入学习算法，包含CBOW和Skip-gram两种架构
# 本代码实现Skip-gram（跳字模型）：用中心词预测上下文词
# 核心思想：语义相近的词在上下文中往往可以互换，因此应有相似的向量表示

import math
import os
import random
import torch
from d2l import torch as d2l

# =============================================================================
# 步骤1：下载并加载PTB数据集
# =============================================================================
# PTB（Penn Tree Bank）：经典的文本数据集，常用于语言模型和词嵌入训练
#@save
d2l.DATA_HUB['ptb'] = (d2l.DATA_URL + 'ptb.zip',
                       '319d85e578af0cdc590547f26231e4e31cdf1e42')

#@save
def read_ptb():
    """将PTB数据集加载到文本行的列表中
    
    返回:
        sentences: 句子列表，每个句子是词元（token）列表
                   例如：[['a', 'b', 'c'], ['d', 'e', 'f', 'g'], ...]
    """
    data_dir = d2l.download_extract('ptb')
    # 读取训练集文件
    with open(os.path.join(data_dir, 'ptb.train.txt')) as f:
        raw_text = f.read()
    # split('\n')按行分割，每行是一个句子
    # line.split()将每行按空格分割为词列表
    return [line.split() for line in raw_text.split('\n')]

# 加载数据
sentences = read_ptb()
f'# sentences数: {len(sentences)}'

# 构建词表，只保留出现频率≥10次的词，低频词会增加噪声
vocab = d2l.Vocab(sentences, min_freq=10)
f'vocab size: {len(vocab)}'


# =============================================================================
# 步骤2：下采样高频词（Sub-sampling）
# =============================================================================
# 动机：高频词（如"the", "a", "in"）提供的信息量少，但出现频率极高
# 下采样可以减少训练数据量，同时保留更多有用的低频词信息
# 公式：P(w_i) = max(0, 1 - sqrt(t / f(w_i)))，其中t是阈值，f是词频
#@save
def subsample(sentences, vocab):
    """下采样高频词
    
    参数:
        sentences: 原始句子列表
        vocab: 词表
    返回:
        subsampled: 下采样后的句子列表
        counter: 词频统计
    """
    # 排除未知词元'<unk>'，这些词在词表中没有实际意义
    sentences = [[token for token in line if vocab[token] != vocab.unk]
                 for line in sentences]
    # 统计每个词的出现次数
    counter = d2l.count_corpus(sentences)
    num_tokens = sum(counter.values())  # 总词数

    # 如果在下采样期间保留词元，则返回True
    # 高频词被丢弃的概率高，低频词被保留的概率高
    def keep(token):
        # random.uniform(0, 1)生成[0,1)之间的随机数
        # 词频越高，sqrt(1e-4 / counter[token] * num_tokens)越小，保留概率越低
        return(random.uniform(0, 1) <
               math.sqrt(1e-4 / counter[token] * num_tokens))

    return ([[token for token in line if keep(token)] for line in sentences],
            counter)

# 执行下采样
subsampled, counter = subsample(sentences, vocab)

# 可视化：比较下采样前后的句子长度分布
d2l.show_list_len_pair_hist(
    ['origin', 'subsampled'], '# tokens per sentence',
    'count', sentences, subsampled);


def compare_counts(token):
    """比较特定词下采样前后的数量"""
    return (f'"{token}"的数量：'
            f'之前={sum([l.count(token) for l in sentences])}, '
            f'之后={sum([l.count(token) for l in subsampled])}')

# 高频词"the"被大量丢弃
compare_counts('the')

# 低频词"join"基本保留
compare_counts('join')

# 将下采样后的文本转换为词索引（corpus：语料库）
corpus = [vocab[line] for line in subsampled]
corpus[:3]


# =============================================================================
# 步骤3：提取中心词和上下文词
# =============================================================================
# Skip-gram模型训练需要"中心词-上下文词"对
# 对于每个中心词，从其周围的窗口中随机选择上下文词
#@save
def get_centers_and_contexts(corpus, max_window_size):
    """返回跳元模型中的中心词和上下文词
    
    参数:
        corpus: 词索引列表的列表
        max_window_size: 最大上下文窗口大小
    返回:
        centers: 中心词列表
        contexts: 上下文词列表（每个中心词对应一个上下文词列表）
    """
    centers, contexts = [], []
    for line in corpus:
        # 要形成"中心词-上下文词"对，每个句子至少需要有2个词
        if len(line) < 2:
            continue
        centers += line  # 所有词都作为中心词
        for i in range(len(line)):  # i是中心词位置
            # 随机选择窗口大小（1到max_window_size之间）
            # 随机窗口可以增加训练样本的多样性
            window_size = random.randint(1, max_window_size)
            # 计算窗口左右边界
            indices = list(range(max(0, i - window_size),
                                 min(len(line), i + 1 + window_size)))
            # 从上下文词中排除中心词本身
            indices.remove(i)
            contexts.append([line[idx] for idx in indices])
    return centers, contexts

# 示例：7个词的句子和3个词的句子
tiny_dataset = [list(range(7)), list(range(7, 10))]
print('数据集', tiny_dataset)
for center, context in zip(*get_centers_and_contexts(tiny_dataset, 2)):
    print('中心词', center, '的上下文词是', context)
    
# 提取PTB数据的所有中心词-上下文词对
all_centers, all_contexts = get_centers_and_contexts(corpus, 5)
f'# "中心词-上下文词对"的数量: {sum([len(contexts) for contexts in all_contexts])}'


# =============================================================================
# 步骤4：负采样（Negative Sampling）
# =============================================================================
# 动机：Skip-gram的原始softmax需要计算所有词的预测概率，计算量太大
# 负采样将多分类问题转化为二分类问题：判断一个词是否是真正的上下文词
# 对于每个正样本（真实的上下文词），采样K个负样本（噪声词）

#@save
class RandomGenerator:
    """根据n个采样权重在{1,...,n}中随机抽取
    
    优化：缓存10000个随机样本，避免频繁调用random.choices
    """
    def __init__(self, sampling_weights):
        # 词索引从1开始（0是<unk>未知词）
        self.population = list(range(1, len(sampling_weights) + 1))
        self.sampling_weights = sampling_weights
        self.candidates = []  # 缓存的候选样本
        self.i = 0  # 当前使用位置

    def draw(self):
        """抽取一个样本"""
        if self.i == len(self.candidates):
            # 缓存k个随机采样结果，提高效率
            self.candidates = random.choices(
                self.population, self.sampling_weights, k=10000)
            self.i = 0
        self.i += 1
        return self.candidates[self.i - 1]
    
# 测试随机生成器
generator = RandomGenerator([2, 3, 4])
[generator.draw() for _ in range(10)]

#@save
def get_negatives(all_contexts, vocab, counter, K):
    """返回负采样中的噪声词
    
    参数:
        all_contexts: 所有上下文词列表
        vocab: 词表
        counter: 词频统计
        K: 每个上下文词对应的负样本数量
    返回:
        all_negatives: 每个中心词对应的负样本列表
    """
    # 采样权重：词频的0.75次方（降低高频词的采样概率，增加低频词机会）
    # 0.75是经验值，既能保证高频词被充分采样，又不至于完全主导
    sampling_weights = [counter[vocab.to_tokens(i)]**0.75
                        for i in range(1, len(vocab))]
    all_negatives, generator = [], RandomGenerator(sampling_weights)
    for contexts in all_contexts:
        negatives = []
        # 需要采样 len(contexts) * K 个负样本
        while len(negatives) < len(contexts) * K:
            neg = generator.draw()
            # 噪声词不能是上下文词（正样本）
            if neg not in contexts:
                negatives.append(neg)
        all_negatives.append(negatives)
    return all_negatives

# 为每个中心词生成5个负样本
all_negatives = get_negatives(all_contexts, vocab, counter, 5)


# =============================================================================
# 步骤5：小批量数据准备
# =============================================================================
#@save
def batchify(data):
    """返回带有负采样的跳元模型的小批量样本
    
    由于不同中心词的上下文+负样本数量不同，需要padding对齐
    
    参数:
        data: [(center, context, negative), ...] 列表
    返回:
        centers: (batch_size, 1) 中心词
        contexts_negatives: (batch_size, max_len) 上下文+负样本
        masks: (batch_size, max_len) 掩码（1表示有效，0表示padding）
        labels: (batch_size, max_len) 标签（1表示正样本，0表示负样本）
    """
    # 找出当前batch中最长的contexts+negatives长度
    max_len = max(len(c) + len(n) for _, c, n in data)
    centers, contexts_negatives, masks, labels = [], [], [], []
    for center, context, negative in data:
        cur_len = len(context) + len(negative)
        centers += [center]
        # 将context和negative拼接，并用0填充到max_len
        contexts_negatives += \
            [context + negative + [0] * (max_len - cur_len)]
        # mask：1表示有效位置，0表示padding位置
        masks += [[1] * cur_len + [0] * (max_len - cur_len)]
        # label：1表示正样本（context），0表示负样本（negative）
        labels += [[1] * len(context) + [0] * (max_len - len(context))]
    return (torch.tensor(centers).reshape((-1, 1)), torch.tensor(
        contexts_negatives), torch.tensor(masks), torch.tensor(labels))

# 测试batchify
x_1 = (1, [2, 2], [3, 3, 3, 3])  # center=1, context=[2,2], negative=[3,3,3,3]
x_2 = (1, [2, 2, 2], [3, 3])     # center=1, context=[2,2,2], negative=[3,3]
batch = batchify((x_1, x_2))

names = ['centers', 'contexts_negatives', 'masks', 'labels']
for name, data in zip(names, batch):
    print(name, '=', data)
    
    
# =============================================================================
# 步骤6：封装完整的数据加载流程
# =============================================================================
#@save
def load_data_ptb(batch_size, max_window_size, num_noise_words):
    """下载PTB数据集，然后将其加载到内存中
    
    参数:
        batch_size: 批量大小
        max_window_size: 最大上下文窗口
        num_noise_words: 每个上下文词的负样本数量（K）
    返回:
        data_iter: DataLoader
        vocab: 词表
    """
    num_workers = 0  # Windows环境建议设为0，避免多进程问题
    sentences = read_ptb()
    vocab = d2l.Vocab(sentences, min_freq=10)
    subsampled, counter = subsample(sentences, vocab)
    corpus = [vocab[line] for line in subsampled]
    all_centers, all_contexts = get_centers_and_contexts(
        corpus, max_window_size)
    all_negatives = get_negatives(
        all_contexts, vocab, counter, num_noise_words)

    class PTBDataset(torch.utils.data.Dataset):
        """PTB数据集类"""
        def __init__(self, centers, contexts, negatives):
            assert len(centers) == len(contexts) == len(negatives)
            self.centers = centers
            self.contexts = contexts
            self.negatives = negatives

        def __getitem__(self, index):
            return (self.centers[index], self.contexts[index],
                    self.negatives[index])

        def __len__(self):
            return len(self.centers)

    dataset = PTBDataset(all_centers, all_contexts, all_negatives)

    data_iter = torch.utils.data.DataLoader(
        dataset, batch_size, shuffle=True,
        collate_fn=batchify, num_workers=0)
    return data_iter, vocab

# 加载数据
data_iter, vocab = load_data_ptb(512, 5, 5)
for batch in data_iter:
    for name, data in zip(names, batch):
        print(name, 'shape:', data.shape)
    break

In [ ]:
# =============================================================================
# Word2Vec 词嵌入训练 - 模型训练与词向量应用
# =============================================================================
# 本代码实现Skip-gram模型的训练和词向量应用
# 核心：使用二元交叉熵损失训练，通过中心词预测上下文词和负样本

import math
import torch
from torch import nn
from d2l import torch as d2l

# =============================================================================
# 步骤1：加载数据
# =============================================================================
batch_size, max_window_size, num_noise_words = 512, 5, 5
data_iter, vocab = d2l.load_data_ptb(batch_size, max_window_size,
                                     num_noise_words)

# =============================================================================
# 步骤2：理解词嵌入层
# =============================================================================
# nn.Embedding是词嵌入查找表，将词索引映射为密集向量
# num_embeddings=20：词表大小
# embedding_dim=4：每个词的向量维度
embed = nn.Embedding(num_embeddings=20, embedding_dim=4)
print(f'Parameter embedding_weight ({embed.weight.shape}, '
      f'dtype={embed.weight.dtype})')

# 示例：批量查找词向量
# x是词索引矩阵，每行是一个样本的词序列
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
embed(x)
# 输出shape为(2, 3, 4)：2个样本，每个样本3个词，每个词4维向量


# =============================================================================
# 步骤3：定义Skip-gram前向传播
# =============================================================================
def skip_gram(center, contexts_and_negatives, embed_v, embed_u):
    """Skip-gram前向传播
    
    参数:
        center: 中心词索引，shape (batch_size, 1)
        contexts_and_negatives: 上下文词和负样本索引，shape (batch_size, max_len)
        embed_v: 中心词嵌入层（输入向量）
        embed_u: 上下文词嵌入层（输出向量）
    返回:
        pred: 预测分数，shape (batch_size, 1, max_len)
    
    原理：计算中心词向量与上下文词向量的点积，分数越高表示越可能是上下文词
    """
    # v: 中心词向量，shape (batch_size, 1, embed_size)
    v = embed_v(center)
    # u: 上下文/负样本向量，shape (batch_size, max_len, embed_size)
    u = embed_u(contexts_and_negatives)
    # bmm: 批量矩阵乘法
    # v @ u.T: (batch, 1, embed) @ (batch, embed, max_len) = (batch, 1, max_len)
    pred = torch.bmm(v, u.permute(0, 2, 1))
    return pred


# =============================================================================
# 步骤4：定义带掩码的二元交叉熵损失
# =============================================================================
class SigmoidBCELoss(nn.Module):
    """带掩码的二元交叉熵损失
    
    由于不同样本的上下文+负样本数量不同，需要mask来忽略padding位置
    """
    def __init__(self):
        super().__init__()

    def forward(self, inputs, target, mask=None):
        """
        参数:
            inputs: 预测分数（未经过sigmoid）
            target: 标签（0或1）
            mask: 掩码（1表示有效，0表示padding）
        """
        # binary_cross_entropy_with_logits = sigmoid + BCE
        # weight=mask：对padding位置赋予0权重
        # reduction="none"：返回每个元素的损失，便于后续处理
        out = nn.functional.binary_cross_entropy_with_logits(
            inputs, target, weight=mask, reduction="none")
        return out.mean(dim=1)  # 对每个样本的所有词求平均

loss = SigmoidBCELoss()

# 测试损失函数
pred = torch.tensor([[1.1, -2.2, 3.3, -4.4]] * 2)  # 预测分数
label = torch.tensor([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])  # 标签
mask = torch.tensor([[1, 1, 1, 1], [1, 1, 0, 0]])  # 第2个样本只有2个有效位置
# 计算损失并归一化（除以有效位置数占总位置数的比例）
loss(pred, label, mask) * mask.shape[1] / mask.sum(axis=1)


# 手动验证损失计算
def sigmd(x):
    """计算sigmoid(-x) = 1/(1+exp(x))"""
    return -math.log(1 / (1 + math.exp(-x)))

# 第1个样本：正样本分数1.1，负样本-2.2, 3.3, -4.4
# 损失 = -log(sigmoid(1.1)) - log(sigmoid(2.2)) - log(sigmoid(-3.3)) - log(sigmoid(4.4))
print(f'{(sigmd(1.1) + sigmd(2.2) + sigmd(-3.3) + sigmd(4.4)) / 4:.4f}')
# 第2个样本：正样本分数-2.2（对应label=1的位置），但mask最后两个为0
# 有效位置：第1个位置label=0，第2个位置label=1
# 损失 = -log(sigmoid(-1.1)) - log(sigmoid(-2.2))，然后除以2
print(f'{(sigmd(-1.1) + sigmd(-2.2)) / 2:.4f}')


# =============================================================================
# 步骤5：定义模型
# =============================================================================
# 使用两个独立的嵌入层：
# - embed_v: 中心词嵌入（输入）
# - embed_u: 上下文词嵌入（输出）
# 这种"双线性"设计比单嵌入层表达能力更强
embed_size = 100
net = nn.Sequential(nn.Embedding(num_embeddings=len(vocab),
                                 embedding_dim=embed_size),
                    nn.Embedding(num_embeddings=len(vocab),
                                 embedding_dim=embed_size))


# =============================================================================
# 步骤6：定义训练函数
# =============================================================================
def train(net, data_iter, lr, num_epochs, device=d2l.try_gpu()):
    """训练Word2Vec模型
    
    参数:
        net: 模型（包含两个嵌入层）
        data_iter: 数据迭代器
        lr: 学习率
        num_epochs: 训练轮数
        device: 计算设备
    """
    # 权重初始化：Xavier均匀初始化
    def init_weights(m):
        if type(m) == nn.Embedding:
            nn.init.xavier_uniform_(m.weight)
    net.apply(init_weights)
    net = net.to(device)
    
    # Adam优化器：自适应学习率，适合稀疏梯度（词嵌入训练）
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    
    # 可视化动画器
    animator = d2l.Animator(xlabel='epoch', ylabel='loss',
                            xlim=[1, num_epochs])
    
    # 规范化的损失之和，规范化的损失数
    metric = d2l.Accumulator(2)
    
    for epoch in range(num_epochs):
        timer, num_batches = d2l.Timer(), len(data_iter)
        for i, batch in enumerate(data_iter):
            optimizer.zero_grad()
            
            # 解包batch数据并移至设备
            center, context_negative, mask, label = [
                data.to(device) for data in batch]
            
            # 前向传播：计算预测分数
            # skip_gram返回shape (batch, 1, max_len)，需要reshape匹配label
            pred = skip_gram(center, context_negative, net[0], net[1])
            
            # 计算损失
            # pred.reshape(label.shape)将(512,1,10)变为(512,10)
            # .float()转换为浮点型
            # / mask.sum(axis=1) * mask.shape[1]：按有效位置数归一化
            l = (loss(pred.reshape(label.shape).float(), label.float(), mask)
                     / mask.sum(axis=1) * mask.shape[1])
            
            # 反向传播和参数更新
            l.sum().backward()
            optimizer.step()
            
            # 记录指标
            metric.add(l.sum(), l.numel())
            
            # 定期更新可视化
            if (i + 1) % (num_batches // 5) == 0 or i == num_batches - 1:
                animator.add(epoch + (i + 1) / num_batches,
                             (metric[0] / metric[1],))
        
    print(f'loss {metric[0] / metric[1]:.3f}, '
          f'{metric[1] / timer.stop():.1f} tokens/sec on {str(device)}')


# =============================================================================
# 步骤7：执行训练
# =============================================================================
lr, num_epochs = 0.002, 5
train(net, data_iter, lr, num_epochs)


# =============================================================================
# 步骤8：应用训练好的词向量 - 查找相似词
# =============================================================================
def get_similar_tokens(query_token, k, embed):
    """查找与query_token最相似的k个词
    
    原理：计算词向量间的余弦相似度，相似度越高表示语义越接近
    
    参数:
        query_token: 查询词
        k: 返回最相似的k个词
        embed: 词嵌入层
    """
    W = embed.weight.data  # 获取所有词的嵌入向量
    x = W[vocab[query_token]]  # 查询词的向量
    
    # 计算余弦相似性：(W · x) / (||W|| * ||x||)
    # 增加1e-9以获得数值稳定性，防止除0
    cos = torch.mv(W, x) / torch.sqrt(torch.sum(W * W, dim=1) *
                                      torch.sum(x * x) + 1e-9)
    
    # topk返回最大的k+1个值（包含查询词本身）
    topk = torch.topk(cos, k=k+1)[1].cpu().numpy().astype('int32')
    
    for i in topk[1:]:  # 跳过第一个（查询词本身）
        print(f'cosine sim={float(cos[i]):.3f}: {vocab.to_tokens(i)}')

# 测试：查找与"chip"最相似的3个词
# 预期结果可能包括：computer, processor, microchip等
get_similar_tokens('chip', 3, net[0])